In [ ]:
import os
import pandas as pd
from pathlib import Path

# os.chdir(Path(os.getcwd()).parent)
os.getcwd()

'c:\\Users\\Tomas\\Desktop\\Thesis Stuff\\Survival_Analysis_Thesis\\Coding'

# Data Cleaning and Saving

Produces the interim datasets the rest of the project consumes, via
`DataCleaner.get_clean_data`. Two flavours per segment:

- **Personal | Professional RAW** - merged, one-day users removed, end-year imputed, segmented by user
  type. No cutoff or inactivity filtering. Used for the distributional exploration
  in `01`.

- **All Users RAW** - merged, one-day users removed, end-year imputed. No segmentation. No cutoff or inactivity filtering. Used for the distributional exploration
  in `01`.

- **Personal | Professional FILTERED** - additionally cutoff-filtered, NaN-metadata removed, and truncated
  at the first churn event (adds `churn_adjusted_date`). Used for the interval
  grid and the survival models.


In [4]:
from src.constants import paths_to_files_and_folders as const
from src.data_cleaning import DataCleaner
from src.constants.segments import PERSONAL, PROFESSIONAL
from src.constants.cleaning import DEFAULT_HHI_THRESHOLD, DEFAULT_CAR_SHARE_ABS, DEFAULT_CAR_SHARE_FRACTION

In [5]:
activity_df = pd.read_csv(const.PATH_TO_RAW_ACTIVITY_DATA_1000)
vehicle_df  = pd.read_csv(const.PATH_TO_RAW_VEHICLE_DATA_1000)
data_cleaner = DataCleaner(activity_df, vehicle_df)

# Decided churn threshold (see interval / gap analysis in notebook 01)
CHURN_THRESHOLD_DAYS_PERSONAL = PERSONAL.churn_threshold_days
CHURN_THRESHOLD_DAYS_PROFESSIONAL = PROFESSIONAL.churn_threshold_days

# Shared split criteria
HHI_THRESHOLD       = DEFAULT_HHI_THRESHOLD
CAR_SHARE_ABS       = DEFAULT_CAR_SHARE_ABS
CAR_SHARE_FRACTION  = DEFAULT_CAR_SHARE_FRACTION



## All Users - RAW

In [6]:
merged_all_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=False,
    # return_personal_use_users=True,          
    filter_one_day_users=True,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "merged_all_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Number of one-day-users: 105
Rows before one-day-user filtering: 560583
Rows after one-day-user filtering: 559649
Rows removed: 934


Cleaning complete
Final rows: 559649
Final unique users: 2650

_Step 4_
Saving File to C:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\Data\interim\merged_all_users_raw.csv


## Personal - RAW


In [7]:
personal_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=True,          # personal
    filter_one_day_users=True,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "personal_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Number of one-day-users: 105
Rows before one-day-user filtering: 560583
Rows after one-day-user filtering: 559649
Rows removed: 934

_Step 4_
Rows before user type filtering: 559649
Filtering by personal users!
Rows after user type filtering: 111985


Cleaning complete
Final rows: 111985
Final unique users: 2435

_Step 5_
Saving File to C:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\Data\interim\personal_users_raw.csv


## Professional - RAW


In [8]:
professional_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=False,         # professional
    filter_one_day_users=True,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "professional_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Number of one-day-users: 105
Rows before one-day-user filtering: 560583
Rows after one-day-user filtering: 559649
Rows removed: 934

_Step 4_
Rows before user type filtering: 559649
Filtering by professional users!
Rows after user type filtering: 37774


Cleaning complete
Final rows: 37774
Final unique users: 215

_Step 5_
Saving File to C:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\Data\interim\professional_users_raw.csv


## Personal - FILTERED

Adds cutoff filtering, NaN-metadata removal, and inactivity truncation on top of
the raw pipeline.


In [9]:
personal_filtered = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_inactivity=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=True,          # personal
    filter_one_day_users=True,
    filter_by_set_cutoff_date=True,
    transform_vehicle_end_year_to_present=True,
    filter_nan_vehicle_metadata=True,
    threshold_value=CHURN_THRESHOLD_DAYS_PERSONAL,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "personal_users_filtered.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Number of one-day-users: 105
Rows before one-day-user filtering: 560583
Rows after one-day-user filtering: 559649
Rows removed: 934

_Step 4_
Rows before set cutoff date filtering: 559649
Rows after set cutoff date filtering: 558400
Rows removed: 1249

_Step 5_
Rows before user type filtering: 558400
Filtering by personal users!
Rows after user type filtering: 112130

_Step 6_
Rows before vehicle metadata filtering: 112130
Rows after vehicle metadata filtering: 101732
Rows removed: 10398

_Step 7_
Filtering activity after inactivity threshold: 160 days
Rows be

## Professional - FILTERED


In [10]:
professional_filtered = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_inactivity=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=False,         # professional
    filter_one_day_users=True,
    filter_by_set_cutoff_date=True,
    transform_vehicle_end_year_to_present=True,
    filter_nan_vehicle_metadata=True,
    threshold_value=CHURN_THRESHOLD_DAYS_PROFESSIONAL, 
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "professional_users_filtered.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Number of one-day-users: 105
Rows before one-day-user filtering: 560583
Rows after one-day-user filtering: 559649
Rows removed: 934

_Step 4_
Rows before set cutoff date filtering: 559649
Rows after set cutoff date filtering: 559579
Rows removed: 70

_Step 5_
Rows before user type filtering: 559579
Filtering by professional users!
Rows after user type filtering: 37774

_Step 6_
Rows before vehicle metadata filtering: 37774
Rows after vehicle metadata filtering: 35087
Rows removed: 2687

_Step 7_
Filtering activity after inactivity threshold: 80 days
Rows befor